# Date Controls

This notebook compares each real trigger schedule with two separate controls
under one selected state representation:

1. a **random-date trigger-panel placebo**

Set `REPRESENTATION` to `binary` for trigger flags alone or to `hybrid` for
the continuous indicator columns together with their trigger flags. The same
frozen date assignments are used in both cases, while each representation has
its own model and result namespace.

The controls are deliberately independent. Each section has one explicit mode:
load saved metrics, evaluate saved models, train only missing models, or retrain
all models under a new run identifier. Run-scoped result paths keep these
operations separate.

## 1 — Common setup


In [ ]:
%matplotlib inline
import hashlib
import multiprocessing
import os
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import stats as scipy_stats
from stable_baselines3 import A2C, PPO, SAC

from utils.config import WINDOWS, SEEDS
from utils.common import (
    control_model_run_name,
    get_indicators,
    load_frozen_target_dates,
    make_env_kwargs,
    make_window_data,
    manifest_row_lookup,
    run_prediction_vecnorm_reconstructed,
)
from utils.tables import compute_metrics, compute_trade_stats
from utils.train_workers import train_model_seeded

ALGO_CLS = {"a2c": A2C, "ppo": PPO, "sac": SAC}


In [11]:
# ── Experiment definition ────────────────────────────────────────────────
REPRESENTATION = "hybrid"  # "binary" or "hybrid"
if REPRESENTATION not in {"binary", "hybrid"}:
    raise ValueError(f"Unsupported representation: {REPRESENTATION!r}")

FAMILIES = {
    "moving_average": {
        "treatment_dataset": (
            "flt_mvx2" if REPRESENTATION == "hybrid" else "flt_mvx2_trg"
        ),
        "daily_dataset": "mvx2",
        "trigger_columns": [
            "close_30_sma_xu_close_60_sma",
            "close_30_sma_xd_close_60_sma",
        ],
    },
    "bollinger": {
        "treatment_dataset": (
            "flt_bollx" if REPRESENTATION == "hybrid" else "flt_bollx_trg"
        ),
        "daily_dataset": "bollx",
        "trigger_columns": [
            "close_xu_boll_lb",
            "close_xu_boll_ub",
        ],
    },
}

# Active comparison. RUN_ID identifies its models, logs, and result tables.
WINDOWS_TO_RUN = ["w1", "w4", "w3"]
RUN_ID = "v1"
ALGOS = ["a2c", "ppo", "sac"]
assert SEEDS == [42, 123, 456, 789, 1234], f"SEEDS in config.py changed: {SEEDS}"

VALID_MODES = {"load_results", "evaluate_models", "train_missing", "retrain_all"}
PLACEBO_MODE = "train_missing"
CIRCULAR_MODE = "train_missing"
PLACEBO_TAG_INFIX = (
    "placebo_hybrid" if REPRESENTATION == "hybrid" else "placebo"
)
CIRCULAR_TAG_INFIX = (
    "circshift_hybrid" if REPRESENTATION == "hybrid" else "circshift"
)

N_CONTROL = len(FAMILIES) * len(WINDOWS_TO_RUN) * len(ALGOS) * len(SEEDS)
N_RANDOM = 2 * N_CONTROL
N_CIRCULAR = N_CONTROL
N_COMBINED = N_RANDOM + N_CIRCULAR
N_PAIRED = 2 * N_CONTROL
N_SUMMARY = 2 * len(FAMILIES) * len(WINDOWS_TO_RUN) * len(ALGOS)
RESULT_CONDITIONS = [
    "real_trigger_alignment",
    "random_date_trigger_panel_placebo",
    "circular_shift_placebo",
]
N_REPRESENTATION_COMBINED = 2 * N_COMBINED
N_REPRESENTATION_SUMMARY = (
    2 * len(FAMILIES) * len(WINDOWS_TO_RUN) * len(ALGOS)
    * len(RESULT_CONDITIONS)
)

MIN_CIRCULAR_SHIFT_DAYS = 61
N_WORKERS = 10
GPU_FRAC = round(0.9 / N_WORKERS, 3)
TIMESTEPS = {"a2c": 50_000, "ppo": 50_000, "sac": 50_000}
ALGO_PARAMS = {
    "a2c": {"n_steps": 5, "ent_coef": 0.01, "learning_rate": 0.0007, "device": "cuda"},
    "ppo": {
        "n_steps": 2048,
        "ent_coef": 0.01,
        "learning_rate": 0.00025,
        "batch_size": 128,
        "device": "cuda",
    },
    "sac": {
        "batch_size": 128,
        "buffer_size": 100_000,
        "learning_rate": 0.0001,
        "learning_starts": 100,
        "ent_coef": "auto_0.1",
        "device": "cuda",
    },
}

FINRL_PATH = Path(os.environ["FINRL_PATH"])
TREATMENT_MODELS_DIR = FINRL_PATH / "models" / "dow" / "experiment_1"
LEGACY_MODEL_RUN_IDS = {"paper_w123_v1", "paper_w143_v1"}
if REPRESENTATION == "binary":
    CONTROL_RUN_DIRNAME = (
        "experiment_smrsc_trg_v1"
        if RUN_ID in LEGACY_MODEL_RUN_IDS
        else f"experiment_smrsc_trg_{RUN_ID}"
    )
    RUN_TABLES_GROUP = "experiment_smrsc_trg_runs"
else:
    CONTROL_RUN_DIRNAME = f"experiment_smrsc_hybrid_{RUN_ID}"
    RUN_TABLES_GROUP = "experiment_smrsc_hybrid_runs"
MODELS_DIR = FINRL_PATH / "models" / "dow" / CONTROL_RUN_DIRNAME
LOG_DIR = FINRL_PATH / "logs" / "dow" / CONTROL_RUN_DIRNAME
# The placebo definitions are frozen and shared across training runs.
SCHEDULE_DIR = (
    FINRL_PATH / "results" / "dow" / "experiment_smrsc_trg_v1" / "schedules"
)

RUN_TABLES_DIR = Path("tables_mv2boll") / RUN_TABLES_GROUP / RUN_ID
PLACEBO_RESULTS_FILE = RUN_TABLES_DIR / "seed_metrics_random_placebo.csv"
CIRCULAR_RESULTS_FILE = RUN_TABLES_DIR / "seed_metrics_circular_shift.csv"
COMBINED_RESULTS_FILE = RUN_TABLES_DIR / "seed_metrics_combined.csv"
COMBINED_PAIRED_FILE = RUN_TABLES_DIR / "paired_deltas_combined.csv"
COMBINED_SUMMARY_FILE = RUN_TABLES_DIR / "summary_combined.csv"
BINARY_REFERENCE_RESULTS_FILE = (
    Path("tables_mv2boll")
    / "experiment_smrsc_trg_runs"
    / "v1"
    / "seed_metrics_combined.csv"
)
REPRESENTATION_RESULTS_FILE = (
    RUN_TABLES_DIR / "seed_metrics_binary_hybrid_combined.csv"
)
REPRESENTATION_SUMMARY_FILE = RUN_TABLES_DIR / "summary_binary_hybrid.csv"

CALENDAR_SOURCE_DATASET = "mvx2"
CALENDAR_RAW_COLS = ["date", "tic", "day", "open", "high", "low", "close", "volume"]
SEED_METRICS_COLS = [
    "family", "window", "algorithm", "seed", "condition",
    "return", "daily_sharpe", "max_drawdown", "trades", "retained_dates",
]
KEY_COLS = ["family", "window", "algorithm", "seed", "condition"]
RUNTIME_ACTIVITY = {
    "training_jobs_run": 0,
    "model_evaluations": 0,
}

print("Representation:", REPRESENTATION)
print("Run ID:", RUN_ID)
print("Windows:", WINDOWS_TO_RUN)
print("Random-date placebo mode:", PLACEBO_MODE)
print("Circular-shift mode:", CIRCULAR_MODE)
print("Control models:", MODELS_DIR)
print("Run results:", RUN_TABLES_DIR)


Representation: hybrid
Run ID: v1
Windows: ['w1', 'w4', 'w3']
Random-date placebo mode: train_missing
Circular-shift mode: train_missing
Control models: /home/main/Programming/finrl/models/dow/experiment_smrsc_hybrid_v1
Run results: tables_mv2boll/experiment_smrsc_hybrid_runs/v1


In [12]:
# ── Mode contracts and non-overwriting output ────────────────────────────
def validate_mode(mode: str, label: str) -> str:
    if mode not in VALID_MODES:
        raise ValueError(
            f"{label} mode must be one of {sorted(VALID_MODES)}; got {mode!r}."
        )
    return mode


def require_fresh_retrain_namespace(modes, paths):
    '''Prevent retrain_all from reusing any populated run directory.'''
    if "retrain_all" not in modes:
        return
    occupied = []
    for path in map(Path, paths):
        if path.exists():
            first_file = next((item for item in path.rglob("*") if item.is_file()), None)
            if first_file is not None:
                occupied.append(first_file)
    if occupied:
        raise FileExistsError(
            "retrain_all requires a fresh RUN_ID with empty model, log, and "
            f"result directories; existing files include {occupied[:3]}"
        )


def safe_write_csv(df: pd.DataFrame, path: Path) -> str:
    '''Atomically create a CSV; never replace a different existing result.'''
    import tempfile

    path = Path(path)

    def existing_status():
        existing = pd.read_csv(path)
        try:
            pd.testing.assert_frame_equal(
                existing.reset_index(drop=True),
                df.reset_index(drop=True),
                check_dtype=False,
                check_exact=False,
                rtol=1e-12,
                atol=1e-12,
            )
        except AssertionError as exc:
            raise FileExistsError(
                f"Refusing to overwrite a different existing result: {path}"
            ) from exc
        return "unchanged"

    if path.exists():
        return existing_status()

    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_name = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="",
            prefix=f".{path.name}.",
            suffix=".tmp",
            dir=path.parent,
            delete=False,
        ) as handle:
            tmp_name = handle.name
            df.to_csv(handle, index=False)
        try:
            # A same-filesystem hard link publishes atomically and fails if another
            # process created the destination after the initial existence check.
            os.link(tmp_name, path)
        except FileExistsError:
            return existing_status()
    except Exception:
        raise
    finally:
        if tmp_name is not None and Path(tmp_name).exists():
            Path(tmp_name).unlink()
    return "written"


def validate_seed_metrics(
    df: pd.DataFrame,
    expected_conditions,
    label: str,
) -> pd.DataFrame:
    missing_columns = set(SEED_METRICS_COLS) - set(df.columns)
    if missing_columns:
        raise ValueError(f"{label} is missing columns: {sorted(missing_columns)}")

    result = df[SEED_METRICS_COLS].copy()
    if result[KEY_COLS].isna().any().any():
        raise ValueError(f"{label} contains missing key values")
    duplicated = result.duplicated(KEY_COLS, keep=False)
    if duplicated.any():
        raise ValueError(
            f"{label} contains duplicate keys:\n"
            f"{result.loc[duplicated, KEY_COLS].head()}"
        )

    expected_sets = {
        "family": set(FAMILIES),
        "window": set(WINDOWS_TO_RUN),
        "algorithm": set(ALGOS),
        "seed": set(SEEDS),
        "condition": set(expected_conditions),
    }
    for column, expected in expected_sets.items():
        actual = set(result[column].unique())
        if actual != expected:
            raise ValueError(
                f"{label}: unexpected {column} values; expected {sorted(expected)}, "
                f"got {sorted(actual)}"
            )

    expected_rows = (
        len(FAMILIES) * len(WINDOWS_TO_RUN) * len(ALGOS) * len(SEEDS)
        * len(expected_conditions)
    )
    if len(result) != expected_rows:
        raise ValueError(
            f"{label}: expected {expected_rows} rows, found {len(result)}"
        )

    counts = result.groupby(
        ["family", "window", "algorithm", "condition"]
    )["seed"].nunique()
    if not (counts == len(SEEDS)).all():
        raise ValueError(f"{label}: not every finalized cell has five seeds")

    metric_columns = ["return", "daily_sharpe", "max_drawdown", "trades", "retained_dates"]
    if result[metric_columns].isna().any().any():
        raise ValueError(f"{label} contains missing metric values")

    return result.sort_values(KEY_COLS, ignore_index=True)


def load_validated_metrics(path: Path, expected_conditions, label: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{label} does not exist: {path}. Switch this section to "
            "evaluate_models to create it from saved models."
        )
    return validate_seed_metrics(pd.read_csv(path), expected_conditions, label)


PLACEBO_MODE = validate_mode(PLACEBO_MODE, "Random-date placebo")
CIRCULAR_MODE = validate_mode(CIRCULAR_MODE, "Circular shift")


In [ ]:
# ── Shared data, schedule, model, training, and evaluation helpers ────────
_RUNTIME_DATA_CACHE = None


def materialize_hybrid_schedule(
    frozen_schedule, daily_df, indicators, raw_columns, path,
    expected_tickers, calendar_df,
):
    '''Use frozen target dates with their own coherent daily indicator values.'''
    target_dates = set(frozen_schedule["date"].astype(str).unique())
    expected_tickers = {str(ticker) for ticker in expected_tickers}
    source = daily_df.copy()
    source["date"] = source["date"].astype(str)
    source["tic"] = source["tic"].astype(str)
    columns = list(dict.fromkeys(list(raw_columns) + list(indicators)))
    missing_columns = set(columns) - set(source.columns)
    if missing_columns:
        raise ValueError(f"{path} daily source lacks {sorted(missing_columns)}")

    frame = source[source["date"].isin(target_dates)][columns].copy()
    frame = frame.sort_values(["date", "tic"], ignore_index=True)
    if frame.duplicated(["date", "tic"]).any():
        raise ValueError(f"{path} daily source has duplicate date/ticker rows")
    if set(frame["date"].unique()) != target_dates:
        raise ValueError(f"{path} contains dates absent from the daily hybrid source")
    ticker_panels = frame.groupby("date")["tic"].agg(set)
    if any(tickers != expected_tickers for tickers in ticker_panels):
        raise ValueError(f"{path} has an incomplete or unexpected ticker panel")
    if len(frame) != len(target_dates) * len(expected_tickers):
        raise ValueError(f"{path} has an invalid hybrid panel size")
    if frame[list(indicators)].isna().any().any():
        raise ValueError(f"{path} contains missing hybrid indicator values")

    calendar = calendar_df.copy()
    calendar["date"] = calendar["date"].astype(str)
    calendar["tic"] = calendar["tic"].astype(str)
    calendar = calendar[
        calendar["date"].isin(target_dates)
        & calendar["tic"].isin(expected_tickers)
    ][["date", "tic", "close"]].sort_values(
        ["date", "tic"], ignore_index=True
    )
    if calendar.duplicated(["date", "tic"]).any():
        raise ValueError(f"{path} evaluation calendar has duplicate rows")
    calendar_panels = calendar.groupby("date")["tic"].agg(set)
    if (
        set(calendar["date"].unique()) != target_dates
        or any(tickers != expected_tickers for tickers in calendar_panels)
    ):
        raise ValueError(f"{path} evaluation calendar lacks a complete ticker panel")
    try:
        pd.testing.assert_series_equal(
            frame["close"].reset_index(drop=True),
            calendar["close"].reset_index(drop=True),
            check_names=False, check_dtype=False, check_exact=False,
            rtol=1e-12, atol=1e-12,
        )
    except AssertionError as exc:
        raise ValueError(
            f"{path} hybrid closes differ from the evaluation calendar"
        ) from exc
    frame["day"] = frame["date"].factorize()[0]
    return frame


def expected_state_space(stock_dim, indicators):
    return 1 + stock_dim * (2 + len(indicators))


@contextmanager
def training_namespace_lock(model_dir, label):
    """Allow only one training writer in a model namespace at a time."""
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    lock_path = model_dir / ".training.lock"
    try:
        descriptor = os.open(
            lock_path, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o644
        )
    except FileExistsError as exc:
        raise RuntimeError(
            f"Another training run is already active in {model_dir}. "
            f"Inspect the lock before removing it: {lock_path}"
        ) from exc
    try:
        message = f"pid={os.getpid()} label={label}\n".encode()
        os.write(descriptor, message)
        os.fsync(descriptor)
        yield lock_path
    finally:
        os.close(descriptor)
        lock_path.unlink(missing_ok=True)


def get_runtime_data():
    '''Load market data only when a section trains or evaluates models.'''
    global _RUNTIME_DATA_CACHE
    if _RUNTIME_DATA_CACHE is not None:
        return _RUNTIME_DATA_CACHE

    treatment_names = [cfg["treatment_dataset"] for cfg in FAMILIES.values()]
    treatment_data = make_window_data(
        treatment_names,
        windows=WINDOWS_TO_RUN,
        splits=["train", "test"],
    )
    daily_names = sorted({cfg["daily_dataset"] for cfg in FAMILIES.values()})
    daily_data = make_window_data(
        daily_names,
        windows=WINDOWS_TO_RUN,
        splits=["train", "test"],
    )
    calendar_data = {
        window: {
            split: daily_data[window][CALENDAR_SOURCE_DATASET][split]["df"][CALENDAR_RAW_COLS].copy()
            for split in ["train", "test"]
        }
        for window in WINDOWS_TO_RUN
    }

    coverage_gaps = []
    for family, cfg in FAMILIES.items():
        for window in WINDOWS_TO_RUN:
            for split in ["train", "test"]:
                treatment_dates = set(
                    treatment_data[window][cfg["treatment_dataset"]][split]["df"]["date"].astype(str)
                )
                calendar_dates = set(calendar_data[window][split]["date"].astype(str))
                missing = sorted(treatment_dates - calendar_dates)
                if missing:
                    coverage_gaps.append((family, window, split, missing))
    if coverage_gaps:
        raise AssertionError(
            f"Calendar dataset {CALENDAR_SOURCE_DATASET!r} does not cover all treatment "
            f"dates: {coverage_gaps[:5]}"
        )

    _RUNTIME_DATA_CACHE = treatment_data, calendar_data, daily_data
    return _RUNTIME_DATA_CACHE


def validate_schedule_contents(
    frame, treatment_df, calendar_df, assignment, trigger_columns, raw_columns, path
):
    '''Verify that market rows and trigger panels match the recorded assignment.'''
    assignment = assignment.copy()
    assignment["date"] = assignment["date"].astype(str)
    assignment["source_date"] = assignment["source_date"].astype(str)
    tickers = sorted(treatment_df["tic"].unique())

    market = calendar_df[raw_columns].copy()
    market["date"] = market["date"].astype(str)
    market = market[
        market["date"].isin(set(assignment["date"]))
        & market["tic"].isin(tickers)
    ].merge(assignment, on="date", how="inner")

    panels = treatment_df[["date", "tic"] + list(trigger_columns)].copy()
    panels["date"] = panels["date"].astype(str)
    panels = panels.rename(columns={"date": "source_date"})
    expected = market.merge(panels, on=["source_date", "tic"], how="inner")
    expected = expected.drop(columns="source_date").sort_values(
        ["date", "tic"], ignore_index=True
    )
    expected["day"] = expected["date"].factorize()[0]
    columns = list(raw_columns) + list(trigger_columns)
    actual = frame[columns].sort_values(["date", "tic"], ignore_index=True)
    try:
        pd.testing.assert_frame_equal(
            actual, expected[columns], check_dtype=False, check_exact=False,
            rtol=1e-12, atol=1e-12,
        )
    except AssertionError as exc:
        raise ValueError(
            f"{path} does not match its recorded market rows and trigger panels"
        ) from exc


def load_saved_schedule_set(prefix: str, manifest_filename: str, required_train_keys=None):
    '''Load every test schedule and only the training schedules needed now.'''
    treatment_data, calendar_data, daily_data = get_runtime_data()
    required_train_keys = set(required_train_keys or [])
    manifest_path = SCHEDULE_DIR / manifest_filename
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing frozen schedule manifest: {manifest_path}")
    manifest = pd.read_csv(manifest_path)
    manifest = manifest[manifest["window"].isin(WINDOWS_TO_RUN)].copy()
    expected_manifest_rows = len(FAMILIES) * len(WINDOWS_TO_RUN) * 2 * len(SEEDS)
    if len(manifest) != expected_manifest_rows:
        raise ValueError(
            f"{manifest_path} should contain {expected_manifest_rows} selected rows, "
            f"found {len(manifest)}"
        )

    schedules = {}
    loaded_files = 0
    for family, cfg in FAMILIES.items():
        schedules[family] = {}
        for window in WINDOWS_TO_RUN:
            schedules[family][window] = {}
            for split in ["train", "test"]:
                schedules[family][window][split] = {}
                treatment_df = treatment_data[window][cfg["treatment_dataset"]][split]["df"]
                treatment_dates = set(treatment_df["date"].astype(str).unique())
                for seed in SEEDS:
                    if split == "train" and (family, window, seed) not in required_train_keys:
                        continue
                    path = SCHEDULE_DIR / f"{prefix}_{family}_{window}_{split}_s{seed}.csv"
                    mrow = manifest_row_lookup(manifest, family, window, split, seed)
                    frozen_target_dates = load_frozen_target_dates(
                        prefix, path, mrow
                    )
                    calendar_df = calendar_data[window][split]
                    if int(mrow["retained_date_count"]) != len(treatment_dates):
                        raise ValueError(f"{path} retained-date count differs from treatment")
                    if prefix == "placebo":
                        source_dates = np.sort(
                            treatment_df["date"].astype(str).unique()
                        )
                        n_tickers = treatment_df["tic"].nunique()
                        complete_dates = (
                            calendar_df.groupby("date")["tic"].count()
                            .pipe(lambda values: values[values == n_tickers])
                            .index.astype(str).to_numpy()
                        )
                        rng = np.random.default_rng(int(mrow["schedule_seed"]))
                        random_dates = rng.choice(
                            complete_dates, size=len(source_dates), replace=False
                        )
                        shuffled_sources = rng.permutation(source_dates)
                        assignment = pd.DataFrame(
                            {"date": random_dates, "source_date": shuffled_sources}
                        )
                        checksum_key = "|".join(
                            f"{target}<-{source}"
                            for target, source in sorted(
                                zip(random_dates, shuffled_sources),
                                key=lambda pair: str(pair[0]),
                            )
                        )
                        checksum = hashlib.sha256(checksum_key.encode()).hexdigest()[:16]
                        if checksum != str(mrow["checksum"]):
                            raise ValueError(f"{path} assignment checksum differs")
                        generated_dates = set(str(date) for date in random_dates)
                        if frozen_target_dates != generated_dates:
                            raise ValueError(f"{path} does not match its saved manifest dates")
                        expected_dates = frozen_target_dates
                    elif prefix == "circshift":
                        complete_dates = np.sort(
                            calendar_df.groupby("date")["tic"].count()
                            .pipe(lambda values: values[values == treatment_df["tic"].nunique()])
                            .index.astype(str)
                            .to_numpy()
                        )
                        if int(mrow["calendar_size"]) != len(complete_dates):
                            raise ValueError(f"{path} calendar size differs from its manifest")
                        offset = int(mrow["offset"])
                        circular_distance = min(offset, len(complete_dates) - offset)
                        if circular_distance != int(mrow["circular_distance"]):
                            raise ValueError(f"{path} circular distance differs from its manifest")
                        if circular_distance < MIN_CIRCULAR_SHIFT_DAYS:
                            raise ValueError(f"{path} violates the minimum circular distance")
                        position_of = {date: index for index, date in enumerate(complete_dates)}
                        treatment_positions = np.array(
                            [position_of[date] for date in sorted(treatment_dates)]
                        )
                        source_dates = np.array(sorted(treatment_dates))
                        shifted_dates = complete_dates[
                            (treatment_positions + offset) % len(complete_dates)
                        ]
                        generated_dates = set(str(date) for date in shifted_dates)
                        if frozen_target_dates != generated_dates:
                            raise ValueError(
                                f"{path} target dates differ from the frozen schedule"
                            )
                        expected_dates = frozen_target_dates
                        assignment = pd.DataFrame(
                            {"date": shifted_dates, "source_date": source_dates}
                        )
                    else:
                        raise ValueError(f"Unsupported schedule prefix: {prefix}")

                    if REPRESENTATION == "hybrid":
                        indicators = get_indicators(cfg["treatment_dataset"])
                        daily_df = daily_data[window][cfg["daily_dataset"]][split]["df"]
                        frozen_dates = pd.DataFrame(
                            {"date": sorted(expected_dates)}
                        )
                        frame = materialize_hybrid_schedule(
                            frozen_dates, daily_df, indicators,
                            CALENDAR_RAW_COLS, path,
                            expected_tickers=set(treatment_df["tic"].astype(str)),
                            calendar_df=calendar_df,
                        )
                    else:
                        if not path.exists():
                            raise FileNotFoundError(f"Missing frozen schedule: {path}")
                        frame = pd.read_csv(path).sort_values(
                            ["date", "tic"], ignore_index=True
                        )
                        frame["date"] = frame["date"].astype(str)
                        frame["day"] = frame["date"].factorize()[0]
                        expected_columns = set(
                            CALENDAR_RAW_COLS + cfg["trigger_columns"]
                        )
                        if set(frame.columns) != expected_columns:
                            raise ValueError(
                                f"{path} has unexpected columns: "
                                f"{sorted(set(frame.columns) ^ expected_columns)}"
                            )
                        ticker_counts = set(
                            frame.groupby("date")["tic"].count().unique()
                        )
                        if ticker_counts != {treatment_df["tic"].nunique()}:
                            raise ValueError(f"{path} has incomplete ticker panels")
                        schedule_dates = set(frame["date"].unique())
                        if schedule_dates != expected_dates:
                            raise ValueError(f"{path} differs from its frozen manifest")
                        validate_schedule_contents(
                            frame, treatment_df, calendar_df, assignment,
                            cfg["trigger_columns"], CALENDAR_RAW_COLS, path,
                        )

                    schedules[family][window][split][seed] = frame
                    loaded_files += 1

    print(
        f"Validated {loaded_files} required frozen {prefix} schedules "
        f"for {WINDOWS_TO_RUN}; wrote none."
    )
    return schedules, manifest


def treatment_run_name(treatment_data, family, window, algorithm, seed):
    dataset = FAMILIES[family]["treatment_dataset"]
    tag = treatment_data[window][dataset]["train"]["tag"]
    return f"{tag}_{algorithm}_s{seed}", TREATMENT_MODELS_DIR


def control_run_name(treatment_data, family, window, algorithm, seed, tag_infix):
    # Tag-string logic now lives in utils.control_model_run_name -- shared with
    # dow_4_advtest 2x2.ipynb, which resolves the same control models for its
    # random-schedule cross-eval. This wrapper only resolves this notebook's own
    # globals (FAMILIES/MODELS_DIR) and keeps the original call signature.
    dataset = FAMILIES[family]["treatment_dataset"]
    treatment_tag = treatment_data[window][dataset]["train"]["tag"]
    run_name = control_model_run_name(treatment_tag, dataset, tag_infix, family, algorithm, seed)
    return run_name, MODELS_DIR


def _require_complete_artifact_pair(model_dir: Path, run_name: str):
    model_path = model_dir / f"{run_name}.zip"
    vecnorm_path = model_dir / f"{run_name}_vecnorm.pkl"
    model_exists = model_path.exists()
    vecnorm_exists = vecnorm_path.exists()
    if model_exists != vecnorm_exists:
        raise RuntimeError(
            f"Incomplete artifact pair for {run_name}: model={model_exists}, "
            f"VecNormalize={vecnorm_exists}. Resolve the partial run manually."
        )
    return model_path, vecnorm_path, model_exists


def expected_control_run_names(tag_infix: str):
    treatment_data, _, _ = get_runtime_data()
    return [
        control_run_name(treatment_data, family, window, algorithm, seed, tag_infix)[0]
        for family in FAMILIES
        for window in WINDOWS_TO_RUN
        for algorithm in ALGOS
        for seed in SEEDS
    ]


def training_schedule_keys(tag_infix: str, target_run_names):
    treatment_data, _, _ = get_runtime_data()
    targets = set(target_run_names)
    keys = set()
    for family in FAMILIES:
        for window in WINDOWS_TO_RUN:
            for seed in SEEDS:
                names = {
                    control_run_name(
                        treatment_data, family, window, algorithm, seed, tag_infix
                    )[0]
                    for algorithm in ALGOS
                }
                if names & targets:
                    keys.add((family, window, seed))
    return keys


def require_treatment_models():
    treatment_data, _, _ = get_runtime_data()
    missing = []
    for family in FAMILIES:
        for window in WINDOWS_TO_RUN:
            for algorithm in ALGOS:
                for seed in SEEDS:
                    run_name, model_dir = treatment_run_name(
                        treatment_data, family, window, algorithm, seed
                    )
                    _, _, exists = _require_complete_artifact_pair(model_dir, run_name)
                    if not exists:
                        missing.append(run_name)
    if missing:
        raise FileNotFoundError(
            f"Treatment preflight: {len(missing)} model pairs are missing; "
            f"first examples: {missing[:3]}"
        )
    print(f"Treatment preflight: found all {N_CONTROL} model pairs.")


def plan_control_artifacts(mode: str, model_dir: Path, run_names, label: str):
    '''Select training targets without creating or replacing any artifact.'''
    complete = []
    missing = []
    for run_name in run_names:
        _, _, exists = _require_complete_artifact_pair(Path(model_dir), run_name)
        (complete if exists else missing).append(run_name)

    if mode == "evaluate_models" and missing:
        raise FileNotFoundError(
            f"{label}: {len(missing)} model pairs are missing. Use train_missing "
            "or select a run that already contains them."
        )
    if mode == "retrain_all" and complete:
        raise FileExistsError(
            f"{label}: retrain_all requires a new RUN_ID; {len(complete)} target "
            f"pairs already exist in {model_dir}."
        )

    targets = (
        list(run_names) if mode == "retrain_all"
        else missing if mode == "train_missing"
        else []
    )
    return {"complete": complete, "missing": missing, "targets": targets}


def build_control_jobs(schedules, tag_infix: str, target_run_names):
    treatment_data, _, _ = get_runtime_data()
    targets = set(target_run_names)
    jobs = []
    for family, cfg in FAMILIES.items():
        dataset = cfg["treatment_dataset"]
        for window in WINDOWS_TO_RUN:
            treatment_tag = treatment_data[window][dataset]["train"]["tag"]
            control_tag = treatment_tag.replace(dataset, f"{tag_infix}_{family}")
            for seed in SEEDS:
                selected_algorithms = [
                    algorithm for algorithm in ALGOS
                    if f"{control_tag}_{algorithm}_s{seed}" in targets
                ]
                if not selected_algorithms:
                    continue

                train_df = schedules[family][window]["train"][seed].copy()
                train_df = train_df.sort_values(["date", "tic"], ignore_index=True)
                train_df.index = train_df["date"].factorize()[0]
                indicators = get_indicators(dataset, df=train_df)
                env_kwargs = make_env_kwargs(train_df, indicators)
                expected_dimension = expected_state_space(
                    train_df["tic"].nunique(), indicators
                )
                if env_kwargs["state_space"] != expected_dimension:
                    raise AssertionError(
                        f"Expected state dimension {expected_dimension}, got "
                        f"{env_kwargs['state_space']}"
                    )

                for algorithm in selected_algorithms:
                    model_name = f"{control_tag}_{algorithm}"
                    jobs.append(
                        (
                            train_df, indicators, algorithm, ALGO_PARAMS[algorithm],
                            TIMESTEPS[algorithm], model_name, GPU_FRAC, seed, None,
                            MODELS_DIR, LOG_DIR,
                        )
                    )
    if len(jobs) != len(targets):
        raise AssertionError(
            f"Constructed {len(jobs)} jobs for {len(targets)} selected targets"
        )
    return jobs


def train_control_models(context):
    initial_plan = plan_control_artifacts(
        context["mode"], MODELS_DIR, context["run_names"], context["label"]
    )
    targets = initial_plan["targets"]
    if not targets:
        print(
            f"{context['label']}: all {N_CONTROL} model pairs are available; "
            "trained none."
        )
        return

    with training_namespace_lock(MODELS_DIR, context["label"]):
        # Recheck under the exclusive lock before submitting any worker.
        artifact_plan = plan_control_artifacts(
            context["mode"], MODELS_DIR, context["run_names"],
            context["label"],
        )
        context["artifact_plan"] = artifact_plan
        targets = artifact_plan["targets"]
        if not targets:
            print(f"{context['label']}: targets became complete; trained none.")
            return

        jobs = build_control_jobs(
            context["schedules"], context["tag_infix"], targets
        )
        process_context = multiprocessing.get_context("spawn")
        with ProcessPoolExecutor(
            max_workers=N_WORKERS, mp_context=process_context
        ) as executor:
            futures = [executor.submit(train_model_seeded, *job) for job in jobs]
            for index, future in enumerate(as_completed(futures), start=1):
                print(f"[{index}/{len(jobs)}] {future.result()}")
        RUNTIME_ACTIVITY["training_jobs_run"] += len(jobs)

        plan_control_artifacts(
            "evaluate_models", MODELS_DIR, context["run_names"],
            context["label"],
        )


def evaluate_condition_rows(condition: str, schedules=None, tag_infix=None):
    treatment_data, calendar_data, _ = get_runtime_data()
    rows = []
    completed = 0
    total = len(FAMILIES) * len(WINDOWS_TO_RUN) * len(ALGOS) * len(SEEDS)

    for family, cfg in FAMILIES.items():
        dataset = cfg["treatment_dataset"]
        for window in WINDOWS_TO_RUN:
            full_df = calendar_data[window]["test"]
            tickers = sorted(full_df["tic"].unique())
            full_price_wide = full_df.pivot(
                index="date", columns="tic", values="close"
            )[tickers]
            full_price_wide.index = pd.to_datetime(full_price_wide.index)
            full_dates = full_price_wide.index

            for algorithm in ALGOS:
                for seed in SEEDS:
                    if condition == "real_trigger_alignment":
                        run_name, model_dir = treatment_run_name(
                            treatment_data, family, window, algorithm, seed
                        )
                        sparse_df = treatment_data[window][dataset]["test"]["df"].copy()
                    else:
                        run_name, model_dir = control_run_name(
                            treatment_data, family, window, algorithm, seed, tag_infix
                        )
                        sparse_df = schedules[family][window]["test"][seed].copy()

                    model_path, vecnorm_path, exists = _require_complete_artifact_pair(
                        model_dir, run_name
                    )
                    if not exists:
                        raise FileNotFoundError(
                            f"Missing saved model pair required for evaluation: {run_name}"
                        )

                    indicators = get_indicators(dataset, df=sparse_df)
                    curve, positions = run_prediction_vecnorm_reconstructed(
                        model_path,
                        vecnorm_path,
                        ALGO_CLS[algorithm],
                        sparse_df,
                        indicators,
                        full_price_wide,
                        full_dates,
                    )
                    metrics = compute_metrics(curve)
                    trades = compute_trade_stats(positions)["total_trades"]
                    rows.append(
                        {
                            "family": family,
                            "window": window,
                            "algorithm": algorithm,
                            "seed": seed,
                            "condition": condition,
                            "return": metrics["Return%"],
                            "daily_sharpe": metrics["Sharpe"],
                            "max_drawdown": metrics["MaxDD%"],
                            "trades": trades,
                            "retained_dates": sparse_df["date"].nunique(),
                        }
                    )
                    completed += 1
                    RUNTIME_ACTIVITY["model_evaluations"] += 1
                    if completed % 10 == 0 or completed == total:
                        print(f"{condition}: evaluated {completed}/{total} saved models")

    return validate_seed_metrics(
        pd.DataFrame(rows, columns=SEED_METRICS_COLS),
        [condition],
        condition,
    )


def preflight_control_experiment(
    *, label, mode, results_file, conditions, schedule_prefix,
    manifest_filename, tag_infix, control_condition, include_treatment,
):
    '''Validate every input before either experiment may start training.'''
    if mode == "load_results":
        metrics = load_validated_metrics(results_file, conditions, label)
        print(f"{label} preflight: loaded {len(metrics)} active-run rows.")
        return {
            "label": label, "mode": mode, "results_file": results_file,
            "conditions": conditions, "loaded_metrics": metrics,
        }

    if include_treatment:
        require_treatment_models()
    run_names = expected_control_run_names(tag_infix)
    artifact_plan = plan_control_artifacts(mode, MODELS_DIR, run_names, label)
    required_train_keys = training_schedule_keys(tag_infix, artifact_plan["targets"])
    schedules, manifest = load_saved_schedule_set(
        schedule_prefix, manifest_filename, required_train_keys
    )
    print(
        f"{label} preflight: {len(artifact_plan['complete'])}/{N_CONTROL} complete, "
        f"{len(artifact_plan['targets'])} selected for training."
    )
    return {
        "label": label, "mode": mode, "results_file": results_file,
        "conditions": conditions, "schedules": schedules,
        "manifest": manifest, "tag_infix": tag_infix,
        "control_condition": control_condition,
        "include_treatment": include_treatment, "run_names": run_names,
        "artifact_plan": artifact_plan,
    }


def run_control_experiment(context):
    '''Load or produce one experiment's metrics using its preflight context.'''
    if context["mode"] == "load_results":
        return context["loaded_metrics"]

    train_control_models(context)
    frames = [
        evaluate_condition_rows(
            context["control_condition"],
            context["schedules"],
            context["tag_infix"],
        )
    ]
    if context["include_treatment"]:
        frames.insert(0, evaluate_condition_rows("real_trigger_alignment"))
    metrics = validate_seed_metrics(
        pd.concat(frames, ignore_index=True), context["conditions"], context["label"]
    )
    write_status = safe_write_csv(metrics, context["results_file"])
    print(
        f"{context['label']} result: {write_status} -> {context['results_file']}"
    )
    return metrics


In [14]:
# ── Global preflight: both experiments are checked before any training ──
require_fresh_retrain_namespace(
    [PLACEBO_MODE, CIRCULAR_MODE], [MODELS_DIR, LOG_DIR, RUN_TABLES_DIR]
)
PLACEBO_CONTEXT = preflight_control_experiment(
    label="Random-date placebo",
    mode=PLACEBO_MODE,
    results_file=PLACEBO_RESULTS_FILE,
    conditions=["real_trigger_alignment", "random_date_trigger_panel_placebo"],
    schedule_prefix="placebo",
    manifest_filename="schedule_manifest.csv",
    tag_infix=PLACEBO_TAG_INFIX,
    control_condition="random_date_trigger_panel_placebo",
    include_treatment=True,
)
CIRCULAR_CONTEXT = preflight_control_experiment(
    label="Circular shift",
    mode=CIRCULAR_MODE,
    results_file=CIRCULAR_RESULTS_FILE,
    conditions=["circular_shift_placebo"],
    schedule_prefix="circshift",
    manifest_filename="circular_shift_manifest.csv",
    tag_infix=CIRCULAR_TAG_INFIX,
    control_condition="circular_shift_placebo",
    include_treatment=False,
)


Treatment preflight: found all 90 model pairs.
Validated 60 required frozen placebo schedules for ['w1', 'w4', 'w3']; wrote none.
Random-date placebo preflight: 0/90 complete, 90 selected for training.
Validated 60 required frozen circshift schedules for ['w1', 'w4', 'w3']; wrote none.
Circular shift preflight: 0/90 complete, 90 selected for training.


## 2 — Random-date placebo

This control keeps the number of retained dates but assigns the schedule to
randomly selected real market dates. Binary runs use the frozen trigger panels;
hybrid runs use the prices and indicator values observed on each selected target
date. Metrics are read from and written to the active `RUN_ID` directory.
The constructor below is reference-only; normal modes load frozen target dates
and never regenerate or overwrite schedules.

- `load_results`: load this run's treatment/placebo metrics.
- `evaluate_models`: evaluate complete saved treatment and placebo models.
- `train_missing`: train only absent placebo model pairs, then evaluate.
- `retrain_all`: require a new `RUN_ID`, train all placebo pairs, then evaluate.


In [15]:
def make_schedule_seed(family: str, window: str, split: str, model_seed: int) -> int:
    '''
    Deterministic schedule seed, independent of the RL model's own training RNG,
    but reproducible from (family, window, split, model_seed) alone. Algo is
    deliberately NOT part of the key, so a2c/ppo/sac share the identical schedule
    for a given family/window/seed, as required for a clean paired comparison.
    '''
    key = f"schedule|{family}|{window}|{split}|{model_seed}"
    digest = hashlib.sha256(key.encode()).hexdigest()
    return int(digest[:8], 16)


def build_placebo_split(treatment_df: pd.DataFrame, calendar_df: pd.DataFrame,
                         trigger_cols: list, schedule_seed: int):
    '''
    Reference-only constructor: random-date rows + shuffled binary
    trigger panels. Each 30-stock trigger panel is moved as a whole block (never
    shuffled ticker-by-ticker); the random dates use their own real prices, not
    the treatment dates' prices. Fully vectorized (merge-based) -- a naive
    per-date Python loop over ~1000-4000 dates x 80 (family/window/split/seed)
    calls takes 6-10+ minutes; this takes ~3s total.

    Returns (placebo_df, manifest_row_dict).
    '''
    rng = np.random.default_rng(schedule_seed)
    tickers = sorted(treatment_df["tic"].unique())
    n_tickers = len(tickers)

    treat_dates = np.sort(treatment_df["date"].unique())
    n_target = len(treat_dates)

    complete_dates = (
        calendar_df.groupby("date")["tic"].count()
        .pipe(lambda s: s[s == n_tickers])
        .index.to_numpy()
    )
    if len(complete_dates) < n_target:
        raise ValueError(
            f"Not enough complete calendar dates ({len(complete_dates)}) "
            f"to match {n_target} treatment dates."
        )
    random_dates = rng.choice(complete_dates, size=n_target, replace=False)
    shuffled_source_dates = rng.permutation(treat_dates)  # panel provenance, shuffled

    assignment = pd.DataFrame({"date": random_dates, "source_date": shuffled_source_dates})

    cal_sel = calendar_df[calendar_df["date"].isin(set(random_dates)) & calendar_df["tic"].isin(tickers)]
    cal_sel = cal_sel.merge(assignment, on="date", how="inner")

    panel_long = treatment_df[["date", "tic"] + trigger_cols].rename(columns={"date": "source_date"})
    placebo_df = cal_sel.merge(panel_long, on=["source_date", "tic"], how="inner")
    placebo_df = placebo_df.drop(columns=["source_date"]).sort_values(["date", "tic"], ignore_index=True)
    placebo_df["day"] = placebo_df["date"].factorize()[0]

    checksum_key = "|".join(
        f"{rd}<-{sd}" for rd, sd in sorted(zip(random_dates, shuffled_source_dates), key=lambda x: str(x[0]))
    )
    checksum = hashlib.sha256(checksum_key.encode()).hexdigest()[:16]

    manifest_row = {
        "schedule_seed": schedule_seed,
        "retained_date_count": n_target,
        "selected_random_dates": ";".join(sorted(str(d) for d in random_dates)),
        "checksum": checksum,
    }
    return placebo_df, manifest_row

In [19]:
placebo_metrics_df = run_control_experiment(PLACEBO_CONTEXT)

display(
    placebo_metrics_df.groupby(["family", "window", "algorithm", "condition"])
    .agg(return_mean=("return", "mean"), sharpe_mean=("daily_sharpe", "mean"), seeds=("seed", "nunique"))
    .reset_index()
)


Random-date placebo: all 90 model pairs are available; trained none.
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
random_date_trigger_panel_placebo: evaluated 10/90 saved models
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
Stock Dimension: 30, State Space: 181
random_date_trigger_panel_placebo: evaluated 20/90 saved models
Stock Dimension: 30, State Space: 181
Stock

,family,window,algorithm,condition,return_mean,sharpe_mean,seeds
0,bollinger,w1,a2c,random_date_trigger_panel_placebo,39.332,1.4578,5
1,bollinger,w1,a2c,real_trigger_alignment,33.664,1.2594,5
2,bollinger,w1,ppo,random_date_trigger_panel_placebo,28.632,1.9852,5
3,bollinger,w1,ppo,real_trigger_alignment,21.110,1.6034,5
4,bollinger,w1,sac,random_date_trigger_panel_placebo,39.084,1.5728,5
5,bollinger,w1,sac,real_trigger_alignment,31.644,1.3884,5
6,bollinger,w3,a2c,random_date_trigger_panel_placebo,14.782,0.8444,5
7,bollinger,w3,a2c,real_trigger_alignment,14.624,0.8616,5
8,bollinger,w3,ppo,random_date_trigger_panel_placebo,15.004,0.9780,5
9,bollinger,w3,ppo,real_trigger_alignment,10.478,0.6218,5


## 3 — Combined paired comparison

The combined analysis joins the active run's treatment/random-placebo metrics
with its circular-only metrics. Differences are always treatment minus the named
control, paired by family, window, algorithm, and model seed.


In [21]:
PAIRED_COLS = [
    "family", "window", "algorithm", "seed", "vs_condition",
    "return_diff", "sharpe_diff", "drawdown_diff",
]
CONTROL_CONDITIONS = [
    "random_date_trigger_panel_placebo",
    "circular_shift_placebo",
]
METRICS = {
    "return": ("return", "return_diff"),
    "daily_sharpe": ("daily_sharpe", "sharpe_diff"),
    "max_drawdown": ("max_drawdown", "drawdown_diff"),
}


def combine_representation_metrics(binary_metrics, hybrid_metrics):
    """Combine validated binary and hybrid results without mixing labels."""
    frames = []
    for representation, metrics in (
        ("binary", binary_metrics),
        ("hybrid", hybrid_metrics),
    ):
        missing = sorted(set(SEED_METRICS_COLS) - set(metrics.columns))
        if missing:
            raise ValueError(
                f"{representation.title()} metrics lack columns: {missing}"
            )
        if len(metrics) != N_COMBINED:
            raise ValueError(
                f"Expected {N_COMBINED} {representation} rows, "
                f"found {len(metrics)}."
            )
        frame = metrics[SEED_METRICS_COLS].copy()
        frame.insert(0, "representation", representation)
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True)
    representation_keys = ["representation"] + KEY_COLS
    duplicate_mask = combined.duplicated(representation_keys, keep=False)
    if duplicate_mask.any():
        duplicate_rows = combined.loc[duplicate_mask, representation_keys]
        raise ValueError(
            "Duplicate representation rows found:\n"
            f"{duplicate_rows.to_string(index=False)}"
        )
    return combined.sort_values(representation_keys, ignore_index=True)


def build_representation_summary(combined):
    """Summarize each representation without pooling seeds or conditions."""
    group_columns = [
        "representation", "family", "window", "algorithm", "condition",
    ]
    summary = (
        combined.groupby(group_columns, as_index=False)
        .agg(
            return_mean=("return", "mean"),
            return_std=("return", "std"),
            daily_sharpe_mean=("daily_sharpe", "mean"),
            daily_sharpe_std=("daily_sharpe", "std"),
            max_drawdown_mean=("max_drawdown", "mean"),
            max_drawdown_std=("max_drawdown", "std"),
            trades_mean=("trades", "mean"),
            retained_dates_mean=("retained_dates", "mean"),
            n_seeds=("seed", "nunique"),
        )
    )
    return summary.sort_values(group_columns, ignore_index=True)


def build_paired_deltas(combined):
    pivot = combined.pivot(
        index=["family", "window", "algorithm", "seed"],
        columns="condition",
        values=["return", "daily_sharpe", "max_drawdown"],
    )
    rows = []
    for (family, window, algorithm, seed), row in pivot.iterrows():
        for control in CONTROL_CONDITIONS:
            rows.append(
                {
                    "family": family,
                    "window": window,
                    "algorithm": algorithm,
                    "seed": seed,
                    "vs_condition": control,
                    "return_diff": (
                        row[("return", "real_trigger_alignment")]
                        - row[("return", control)]
                    ),
                    "sharpe_diff": (
                        row[("daily_sharpe", "real_trigger_alignment")]
                        - row[("daily_sharpe", control)]
                    ),
                    "drawdown_diff": (
                        row[("max_drawdown", "real_trigger_alignment")]
                        - row[("max_drawdown", control)]
                    ),
                }
            )
    result = pd.DataFrame(rows, columns=PAIRED_COLS)
    return result.sort_values(
        ["family", "window", "algorithm", "seed", "vs_condition"],
        ignore_index=True,
    )


def paired_ci(values):
    count = len(values)
    mean = values.mean()
    std = values.std(ddof=1) if count > 1 else np.nan
    if count > 1 and std > 0:
        standard_error = std / np.sqrt(count)
        critical = scipy_stats.t.ppf(0.975, df=count - 1)
        return mean, std, mean - critical * standard_error, mean + critical * standard_error
    return mean, std, np.nan, np.nan


def verdict_from_ci(ci_low, ci_high, control):
    if ci_low > 0:
        return f"Evidence for real-date alignment vs. {control}"
    if ci_high < 0:
        return f"Evidence favoring {control} over real-date alignment"
    return f"No paired Sharpe difference at the 95% level vs. {control}"


def build_summary(combined, paired):
    rows = []
    grouped = paired.groupby(["family", "window", "algorithm", "vs_condition"])
    for (family, window, algorithm, control), group in grouped:
        treatment = combined[
            (combined["family"] == family)
            & (combined["window"] == window)
            & (combined["algorithm"] == algorithm)
            & (combined["condition"] == "real_trigger_alignment")
        ]
        comparison = combined[
            (combined["family"] == family)
            & (combined["window"] == window)
            & (combined["algorithm"] == algorithm)
            & (combined["condition"] == control)
        ]
        result = {
            "family": family,
            "window": window,
            "algorithm": algorithm,
            "vs_condition": control,
            "n_seeds": len(group),
        }
        for metric_name, (seed_column, diff_column) in METRICS.items():
            mean, std, ci_low, ci_high = paired_ci(group[diff_column])
            result[f"{metric_name}_treatment_mean"] = treatment[seed_column].mean()
            result[f"{metric_name}_placebo_mean"] = comparison[seed_column].mean()
            result[f"{metric_name}_paired_diff_mean"] = mean
            result[f"{metric_name}_paired_diff_std"] = std
            result[f"{metric_name}_ci95_lo"] = ci_low
            result[f"{metric_name}_ci95_hi"] = ci_high
            if metric_name == "daily_sharpe":
                result["verdict"] = verdict_from_ci(ci_low, ci_high, control)
        rows.append(result)
    return pd.DataFrame(rows).sort_values(
        ["family", "window", "algorithm", "vs_condition"],
        ignore_index=True,
    )


In [23]:
combined_metrics_df = validate_seed_metrics(
    pd.concat([placebo_metrics_df, circular_metrics_df], ignore_index=True),
    RESULT_CONDITIONS,
    "Combined metrics",
)
paired_df = build_paired_deltas(combined_metrics_df)
summary_df = build_summary(combined_metrics_df, paired_df)

if len(paired_df) != N_PAIRED:
    raise AssertionError(f"Expected {N_PAIRED} paired rows, found {len(paired_df)}")
if len(summary_df) != N_SUMMARY:
    raise AssertionError(f"Expected {N_SUMMARY} summary rows, found {len(summary_df)}")
if set(paired_df["vs_condition"]) != set(CONTROL_CONDITIONS):
    raise AssertionError("Paired results do not contain both controls")
if not (summary_df["n_seeds"] == len(SEEDS)).all():
    raise AssertionError("A summary cell is not based on five paired seeds")

combined_write_status = safe_write_csv(combined_metrics_df, COMBINED_RESULTS_FILE)
paired_write_status = safe_write_csv(paired_df, COMBINED_PAIRED_FILE)
summary_write_status = safe_write_csv(summary_df, COMBINED_SUMMARY_FILE)

print(f"Combined metrics ({len(combined_metrics_df)} rows): {combined_write_status}")
print(f"Paired deltas ({len(paired_df)} rows): {paired_write_status}")
print(f"Summary ({len(summary_df)} rows): {summary_write_status}")
display(summary_df)


Combined metrics (270 rows): written
Paired deltas (180 rows): written
Summary (36 rows): written


,family,window,algorithm,vs_condition,n_seeds,return_treatment_mean,return_placebo_mean,return_paired_diff_mean,return_paired_diff_std,return_ci95_lo,...,daily_sharpe_paired_diff_std,daily_sharpe_ci95_lo,daily_sharpe_ci95_hi,verdict,max_drawdown_treatment_mean,max_drawdown_placebo_mean,max_drawdown_paired_diff_mean,max_drawdown_paired_diff_std,max_drawdown_ci95_lo,max_drawdown_ci95_hi
0,bollinger,w1,a2c,circular_shift_placebo,5,33.664,31.336,2.328,1.423137,0.560942,...,0.085163,0.049056,0.260544,Evidence for real-date alignment vs. circular_...,-20.040,-23.254,3.214,3.835340,-1.548203,7.976203
1,bollinger,w1,a2c,random_date_trigger_panel_placebo,5,33.664,39.332,-5.668,4.026248,-10.667247,...,0.121490,-0.349250,-0.047550,Evidence favoring random_date_trigger_panel_pl...,-20.040,-15.260,-4.780,5.552301,-11.674093,2.114093
2,bollinger,w1,ppo,circular_shift_placebo,5,21.110,26.638,-5.528,2.031199,-8.050067,...,0.248709,-0.612213,0.005413,No paired Sharpe difference at the 95% level v...,-8.812,-7.962,-0.850,2.098583,-3.455735,1.755735
3,bollinger,w1,ppo,random_date_trigger_panel_placebo,5,21.110,28.632,-7.522,6.738558,-15.889025,...,0.231343,-0.669051,-0.094549,Evidence favoring random_date_trigger_panel_pl...,-8.812,-8.554,-0.258,1.672011,-2.334076,1.818076
4,bollinger,w1,sac,circular_shift_placebo,5,31.644,33.098,-1.454,9.212992,-12.893441,...,0.382193,-0.291756,0.657356,No paired Sharpe difference at the 95% level v...,-12.510,-18.124,5.614,10.736060,-7.716579,18.944579
5,bollinger,w1,sac,random_date_trigger_panel_placebo,5,31.644,39.084,-7.440,15.860544,-27.133466,...,0.537170,-0.851384,0.482584,No paired Sharpe difference at the 95% level v...,-12.510,-13.122,0.612,5.702773,-6.468928,7.692928
6,bollinger,w3,a2c,circular_shift_placebo,5,14.624,18.384,-3.760,7.508592,-13.083148,...,0.344971,-0.623538,0.233138,No paired Sharpe difference at the 95% level v...,-17.106,-17.262,0.156,3.166722,-3.776005,4.088005
7,bollinger,w3,a2c,random_date_trigger_panel_placebo,5,14.624,14.782,-0.158,10.294813,-12.940698,...,0.503880,-0.608450,0.642850,No paired Sharpe difference at the 95% level v...,-17.106,-18.644,1.538,1.449300,-0.261543,3.337543
8,bollinger,w3,ppo,circular_shift_placebo,5,10.478,7.898,2.580,20.934189,-23.413228,...,1.129915,-1.284575,1.521375,No paired Sharpe difference at the 95% level v...,-17.196,-15.898,-1.298,3.103719,-5.151776,2.555776
9,bollinger,w3,ppo,random_date_trigger_panel_placebo,5,10.478,15.004,-4.526,14.235084,-22.201192,...,0.755278,-1.294002,0.581602,No paired Sharpe difference at the 95% level v...,-17.196,-13.986,-3.210,2.246442,-5.999326,-0.420674
